# Attaching Data Assets (raw and processed)
- ophys:
    - Stage_0 (1 session): Natural movies
    - Stage_1 (3 sessions): drifting gratings

- coregistration:
    -   ophys_zstack:
        - Ophys functional (4 sessions, 8 planes) to ophys structure (1 3D image 0-400um) mapping
    - xenium_zstack:
        - Xenium to ophys strucure mapping
- xenium:
    - processed:
        - images
        - segmentation masks
        - cell locations
        - transcript locations
        - cellxgene table
    - cell_types:
        - cell types for each cell (based on ABC atlas)
        - cell typting measures

In [21]:
import codeocean
import codeocean.data_asset
import aind_session
from aind_session.utils.codeocean_utils import get_data_asset_source_dir, get_codeocean_client, search_data_assets, get_data_asset_search_query, get_subject_data_assets
from analysis_utils import get_running_timestamps, get_running_df, get_stimulus_timestamps
from codeocean.data_asset import DataAssetAttachParams
import boto3
import json
import numpy as np
from pathlib import Path
import pandas as pd


import pickle
DATA_DIR = Path("/root/capsule/data")

In [22]:
mouse_ids = [778174,786297,797371]
client = get_codeocean_client()
co_assets_dict = {}
    
for mouse_id in mouse_ids:
    co_assets_dict[mouse_id] = {}
    co_assets_dict[mouse_id]['ophys'] = {}
    assets = list(get_subject_data_assets(mouse_id))
    assets.sort(key=lambda x: x.name)
    # search_params["query"] = get_data_asset_search_query(mouse_id=mouse_id)
    # from_co = search_data_assets(search_params) + search_data_assets(
    #     {"query": str(mouse_id)}
    # )
    assets_raw = [asset for asset in assets if 'multiplane-ophys' in asset.tags and 'raw' in asset.tags and 'multisession' not in asset.tags]

    for asset_raw in assets_raw:
        s3_dir = get_data_asset_source_dir(asset_raw.id).as_posix()
        s3 = boto3.resource('s3')
        BUCKET_NAME = s3_dir.split('/')[2]
        PREFIX = '/'.join(s3_dir.split('/')[3:]) + '/'
        KEY = PREFIX + 'session.json'
        obj = s3.Object(BUCKET_NAME, KEY)
        data = obj.get()['Body'].read()
        json_string = data.decode('utf-8')
        session_json = json.loads(json_string)
        session_type = session_json['session_type']
        if session_type == 'STAGE_0' or session_type == 'STAGE_1':
            asset_processed = [asset for asset in assets if f"{asset_raw.name}_processed" in asset.name][-1]
            co_assets_dict[mouse_id]['ophys'][asset_raw.name] = {'session_type':session_type,'raw':asset_raw,"processed":asset_processed} 

    ophys_asset_names =  list(co_assets_dict[mouse_id]['ophys'].keys())
    co_assets_dict[mouse_id]['ophys'] = {ophys_asset_name:co_assets_dict[mouse_id]['ophys'][ophys_asset_name] for ophys_asset_name in ophys_asset_names[:4]}
    for asset_name in co_assets_dict[mouse_id]['ophys']:
        co_assets_dict[mouse_id]['ophys'][asset_name]['glm'] = [asset for asset in assets if f"{asset_name}_glm" in asset.name][-1]
    
    co_assets_dict[mouse_id]['cortical_zstack'] = {}
    co_assets_dict[mouse_id]['cortical_zstack']['registered'] = [asset for asset in assets if asset.name.startswith(f'ophys-z-stacks_{mouse_id}') and asset.name.endswith('registered')][-1]
    co_assets_dict[mouse_id]['cortical_zstack']['segmented'] = [asset for asset in assets if asset.name.startswith(f'ophys-z-stacks_{mouse_id}') and asset.name.endswith('segmented_cpsam')][-1]
    
    co_assets_dict[mouse_id]['xenium'] = {}
    co_assets_dict[mouse_id]['xenium']['processed'] = [asset for asset in assets if asset.name.startswith("Xenium_") and asset.name.endswith("_processed")][-1]
    co_assets_dict[mouse_id]['xenium']['cell_types'] = [asset for asset in assets if asset.name.startswith("Xenium_") and asset.name.endswith("_mapped")][-1]

    co_assets_dict[mouse_id]['coregistration'] = {}
    co_assets_dict[mouse_id]['coregistration']['ophys_zstack'] = [asset for asset in assets if 'cortical-zstack-to-fov' in asset.tags][-1]
    co_assets_dict[mouse_id]['coregistration']['xenium_zstack'] = [asset for asset in assets if asset.name.startswith("Xenium_ophys") and 'coregistration' in asset.tags][-1]
    

# co_assets = list(np.hstack([[a['raw'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
# [a['processed'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
# [a['glm'] for a in co_assets_dict[mouse_id]['ophys'].values()]+\
# [a for a in co_assets_dict[mouse_id]['xenium'].values()]+\
# [a for a in co_assets_dict[mouse_id]['coregistration'].values()] for mouse_id in mouse_ids]))

# for asset in co_assets:
#     print(asset.id)

In [20]:
co_assets = list(np.hstack([[a for a in co_assets_dict[mouse_id]['cortical_zstack'].values()] for mouse_id in mouse_ids]))
for asset in co_assets:
    print(asset.id)

8905fc2b-93c3-4176-93a3-74e2371f31f4
0c7be3cc-7252-4a63-871a-8af56fb35668
cd10184a-9fd0-4962-a107-cf6f9c516004
b561688b-6e7b-4291-8f5a-b6200425095e
c3b48f98-a0e5-476c-8d05-0f941f8b02d1
1704b827-5c0f-41d0-8dad-a6ae8e3ba4ce


In [24]:

data_info = {}

for mouse_id in mouse_ids:
    data_info[mouse_id] = {}
    data_info[mouse_id]['ophys'] = {}
    data_info[mouse_id]['cortical_zstack'] = {}
    data_info[mouse_id]['xenium'] = {}
    data_info[mouse_id]['coregistration'] = {}


    ophys_asset_names =  list(co_assets_dict[mouse_id]['ophys'].keys())
    data_info[mouse_id]['ophys'] = {f"session_{session_ind}":co_assets_dict[mouse_id]['ophys'][ophys_asset_name].copy() for session_ind, ophys_asset_name in enumerate(ophys_asset_names)}
    for session in data_info[mouse_id]['ophys'].keys():
        data_info[mouse_id]['ophys'][session]['raw'] = data_info[mouse_id]['ophys'][session]['raw'].name
        data_info[mouse_id]['ophys'][session]['processed'] = data_info[mouse_id]['ophys'][session]['processed'].name
        data_info[mouse_id]['ophys'][session]['glm'] = data_info[mouse_id]['ophys'][session]['glm'].name

    data_info[mouse_id]['xenium']['processed'] = co_assets_dict[mouse_id]['xenium']['processed'].name
    data_info[mouse_id]['xenium']['cell_types'] = co_assets_dict[mouse_id]['xenium']['cell_types'].name
    data_info[mouse_id]['coregistration']['ophys_zstack'] = co_assets_dict[mouse_id]['coregistration']['ophys_zstack'].name
    data_info[mouse_id]['coregistration']['xenium_zstack'] = co_assets_dict[mouse_id]['coregistration']['xenium_zstack'].name
    data_info[mouse_id]['cortical_zstack']['registered'] = co_assets_dict[mouse_id]['cortical_zstack']['registered'].name
    data_info[mouse_id]['cortical_zstack']['segmented'] = co_assets_dict[mouse_id]['cortical_zstack']['segmented'].name

data_info
with open('/root/capsule/code/Preprocessing/data_info.pkl', 'wb') as f:
    pickle.dump(data_info, f)


In [8]:
data_info

{778174: {'ophys': {'session_0': {'session_type': 'STAGE_0',
    'raw': 'multiplane-ophys_778174_2025-03-06_12-34-12',
    'processed': 'multiplane-ophys_778174_2025-03-06_12-34-12_processed_2025-03-07_18-51-08',
    'glm': 'multiplane-ophys_778174_2025-03-06_12-34-12_glm_movie_v2'},
   'session_1': {'session_type': 'STAGE_1',
    'raw': 'multiplane-ophys_778174_2025-03-14_11-17-01',
    'processed': 'multiplane-ophys_778174_2025-03-14_11-17-01_processed_2025-03-15_15-57-57',
    'glm': 'multiplane-ophys_778174_2025-03-14_11-17-01_glm_dg_v2'},
   'session_2': {'session_type': 'STAGE_1',
    'raw': 'multiplane-ophys_778174_2025-03-17_10-34-20',
    'processed': 'multiplane-ophys_778174_2025-03-17_10-34-20_processed_2025-03-18_14-58-46',
    'glm': 'multiplane-ophys_778174_2025-03-17_10-34-20_glm_dg_v2'},
   'session_3': {'session_type': 'STAGE_1',
    'raw': 'multiplane-ophys_778174_2025-03-20_11-58-45',
    'processed': 'multiplane-ophys_778174_2025-03-20_11-58-45_processed_2025-03-21_

The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.
